# 11 - LightGCN (graph collaborative filtering)

A graph neural network that learns user/item embeddings by message-passing over the
user-item interaction graph, trained with a **BPR ranking loss**. This is the headline
ranking-oriented extension.

**Requires PyTorch.** On CPU this is slow, so we train on a **user subsample** (a
reduced-scale demonstration); with a GPU you can raise the sample. Embedding dot-products
are *preference scores*, not ratings, so LightGCN is reported on **ranking metrics only**.

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Train LightGCN (BPR loss over the interaction graph)

In [3]:
from hybrid_recsys.models.lightgcn import LightGCNRecommender

# Ensure the ranking-eval users are inside the training graph, then fill up to ~30K users.
keep   = set(int(u) for u in eval_user_ids)
others = np.array([u for u in train["userId"].unique() if int(u) not in keep])
fill   = rng.choice(others, size=min(30_000 - len(keep), len(others)), replace=False)
keep  |= set(int(x) for x in fill)
train_lg = train[train["userId"].isin(keep)]
print(f"LightGCN training on {len(keep):,} users / {len(train_lg):,} interactions")

# Full-batch training: ONE graph propagation per epoch, so epochs are cheap (esp. on GPU).
lg = LightGCNRecommender(dim=64, n_layers=3, epochs=200, lr=5e-3, max_users=None,
                         random_state=RANDOM_STATE)
lg.fit(train_lg)
lg.save()
print("saved lightgcn_model.joblib")

LightGCN training on 30,000 users / 3,635,305 interactions
LightGCN on cuda: 30,000 users, 30,192 items, 3,635,305 interactions
  epoch   1/200  loss=0.6930
  epoch  10/200  loss=0.6125
  epoch  20/200  loss=0.3778
  epoch  30/200  loss=0.2238
  epoch  40/200  loss=0.1804
  epoch  50/200  loss=0.1707
  epoch  60/200  loss=0.1668
  epoch  70/200  loss=0.1641
  epoch  80/200  loss=0.1596
  epoch  90/200  loss=0.1570
  epoch 100/200  loss=0.1545
  epoch 110/200  loss=0.1516
  epoch 120/200  loss=0.1491
  epoch 130/200  loss=0.1462
  epoch 140/200  loss=0.1435
  epoch 150/200  loss=0.1408
  epoch 160/200  loss=0.1382
  epoch 170/200  loss=0.1352
  epoch 180/200  loss=0.1325
  epoch 190/200  loss=0.1297
  epoch 200/200  loss=0.1279
saved lightgcn_model.joblib


## Evaluate — ranking only (embedding scores are not ratings)

In [4]:
from hybrid_recsys.evaluation.metrics import evaluate_ranking_sampled

lg_predict = lambda u, i: lg.predict(u, i)
ranking = evaluate_ranking_sampled(test_sample, lg_predict, train_val,
                                   all_movie_ids=all_movie_ids, n_negatives=N_NEGATIVES,
                                   k_values=[5, 10, 20], random_state=RANDOM_STATE)
m = {"rmse": None, "mae": None,
     **{f"k{k}": {kk: round(vv, 4) for kk, vv in v.items()} for k, v in ranking.items()}}
save_metric("LightGCN", m)
print(f"F1@10={m['k10']['f1']}  (RMSE/MAE omitted - ranking model)")

F1@10=0.6206  (RMSE/MAE omitted - ranking model)


## Ranking metrics @ K

In [5]:
save_fig(ranking_curve(m, "LightGCN"), "eval_lightgcn_ranking")

## Example recommendations for the demo user

In [6]:
seen = set(user_ratings_map.get(demo_user, {}))
cand = rng.choice(all_movie_ids, size=3000, replace=False)
recs = top_n(lg_predict, demo_user, seen, cand, movies, n=10)
display(recs[["clean_title", "genres", "pred"]])

,clean_title,genres,pred
0,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,10.929
1,Thank You for Smoking,Comedy|Drama,10.566
2,Mighty Aphrodite,Comedy|Drama|Romance,9.034
3,How to Train Your Dragon 2,Action|Adventure|Animation,8.548
4,"Player, The",Comedy|Crime|Drama,8.506
5,Despicable Me 2,Animation|Children|Comedy|IMAX,7.886
6,Mickey Blue Eyes,Comedy|Romance,7.429
7,Lara Croft Tomb Raider: The Cradle of Life,Action|Adventure|Comedy|Romance|Thriller,7.406
8,Desperately Seeking Susan,Comedy|Drama|Romance,7.253
9,Logan's Run,Action|Adventure|Sci-Fi,7.122


## Takeaway

LightGCN optimises ranking directly (BPR), so compare its F1@K against the rating-first
models (SVD, Stacked Hybrid) - not their RMSE. Even at reduced scale it should rank
competitively; published ML-25M benchmarks put full LightGCN at the top on NDCG@10.